# 00_config: shared configuration for the spine wearable recovery-debt study

All of Us Research Program, Controlled Tier cdrv9 (resource name `C2025Q4R6`), Verily Workbench 2.0.
Working title: "Cumulative ambulatory activity loss after elective cervical and lumbar spine surgery:
a wearable-linked cohort study in the All of Us Research Program."

**Every later notebook and script inside the perimeter starts with `%run 00_config.ipynb`.**
This file runs **inside the perimeter only**. Local code (figure rendering, table rendering, prose,
docx assembly, verification) reads exported aggregates and never imports anything defined here.

## COMPLIANCE (DUCC section 3.1, AOS-CS section 9): read before running anything

This session is driven through a browser automation that transmits notebook output, screenshots and
DOM, to an external AI model. **No individual-level (participant-level) data may ever be displayed.**
"Displayed" here means rendered anywhere inside this VM, not merely returned to a caller.

- Use `safe_show`, `safe_counts`, `safe_n`, `round20`, `n_pct` and `safe_export`, imported below from
  `disclosure.py`. Never call `df.head()`, never print a row, never print a per-person step count,
  date, or identifier.
- Model-visible output is aggregates only: counts above 20, rounded to the nearest 20, with 1 through
  20 suppressed. A percentage is suppressed whenever its numerator count is suppressed,
  because a percentage times a disclosed denominator recovers the hidden count exactly. Percentages
  are computed from the rounded count and printed to zero decimals.
- Controlled Tier dates are **unshifted**, so any date column and any near-unique column is an
  identifier in an export, not a covariate.
- All modeling on row-level data stays in this VM. No row ever reaches an external API.
- Any raw individual-level inspection is done by the human, with the automation paused.

## COST

This notebook runs on a paid cloud environment and every query bills.

- No query executes without a printed dry-run byte estimate, and every query carries a hard
  `maximum_bytes_billed` cap, so an over-budget query **fails rather than bills**. `q_guarded` below
  is the only query path any later module may use.
- A dry run is free, and it prices the columns referenced rather than the table.
- **Delete the compute environment at the end of every session. Deleted, not paused.** Pausing stops
  the VM charge while the persistent disk keeps billing until the disk itself is deleted; an empty
  Apps tab is the only proof that spend is zero. Notebooks persist in the workspace rather than on
  the compute disk, so deleting the environment loses nothing that was saved.
- The whole-project budget is under $2. `session_cost_report()` below prints the running actual for
  the end-of-session handoff block in `SESSION-LOG.md`.

In [ ]:
from __future__ import annotations

import os
import re
import subprocess

import pandas as pd

# ---- Idempotence guard: a second %run rebinds every name and skips the network work ----
# Cell 0 states that every later notebook begins with `%run 00_config.ipynb`. Without a guard each
# notebook in a session repeats about seven `wb` invocations, two dataset metadata calls, a possible
# dataset creation, and three real BigQuery jobs, none of which can tell it anything the first run
# did not. The guard covers the NETWORK work only: every function and constant defined here is
# rebound on every run, because rebinding is free and because a re-run is how an edit to this file
# is picked up.
#
# To force the full run again, after changing a workspace resource or to re-probe write access:
#     CONFIG_FORCE = True
#     %run 00_config.ipynb
# or export SPINEWEAR_CONFIG_FORCE=1 before starting the kernel. CONFIG_FORCE is ONE-SHOT: the last
# cell clears it, so forcing one notebook does not silently force every later one too.
_CONFIG_CACHED_NAMES = ("WORKSPACE_CDR", "PREP_CDR", "WORKSPACE_BUCKET", "CDR_LOCATION",
                        "WRITE_PROBE_RESULT", "PERSON_N_ROUNDED")
CONFIG_FORCE = bool(globals().get("CONFIG_FORCE", False)) or (
    os.environ.get("SPINEWEAR_CONFIG_FORCE", "").strip().lower() not in ("", "0", "false", "no"))
CONFIG_SKIP_NETWORK = (bool(globals().get("CONFIG_READY", False))
                       and not CONFIG_FORCE
                       and all(name in globals() for name in _CONFIG_CACHED_NAMES))
CONFIG_READY = False   # the last cell sets it True, so a run that raises never counts as configured

# ---- Google project: this workspace's own GCP project, and the project that PAYS for BigQuery ----
# Resolved, never hardcoded: a workspace project is minted per workspace as wb-<random>-NNN.
GOOGLE_PROJECT = (os.environ.get("GOOGLE_PROJECT")
                  or os.environ.get("GOOGLE_CLOUD_PROJECT")
                  or os.environ.get("GCP_PROJECT"))

# A resolved BigQuery dataset looks like project.dataset; a resolved bucket looks like a gs:// path.
# Anything else on stdout is a diagnostic or a usage message, not an id.
_BQ_DATASET_RE = re.compile(r"^[A-Za-z][A-Za-z0-9._:-]*\.[A-Za-z0-9_]+$")
_GCS_PATH_RE = re.compile(r"^gs://[a-z0-9][a-z0-9._-]*(/.*)?$")


def _run_wb(args: list[str], *, timeout: int = 90) -> tuple[int, str, str]:
    """Run one `wb` CLI command. Returns (returncode, stdout, reason), reason empty on success.

    The reason exists so callers can tell the failure modes apart. Collapsing all of them into an
    empty result makes `wb` not installed, a timeout, a permission error and "that resource does not
    exist" indistinguishable, and 01_probe.py is specified to distinguish them.
    """
    try:
        out = subprocess.run(args, capture_output=True, text=True, timeout=timeout)
    except FileNotFoundError:
        return (127, "", "the `wb` CLI is not on PATH in this environment")
    except subprocess.TimeoutExpired:
        return (124, "", f"`wb` did not return within {timeout} seconds")
    except Exception as exc:
        return (125, "", f"`wb` could not be started ({type(exc).__name__}: {exc})")
    if out.returncode != 0:
        detail = ((out.stderr or "").strip() or (out.stdout or "").strip() or "(no message)")
        return (out.returncode, out.stdout or "",
                f"`wb` exited {out.returncode}: {detail.splitlines()[0][:200]}")
    return (0, out.stdout or "", "")


def wb_resolve(name: str, *, expect: str = "any") -> str | None:
    """Resolve a Verily Workbench *referenced resource* (by name) to its cloud id.

    Verily references the CDR and the buckets by NAME (for example 'C2025Q4R6'); this returns the
    fully qualified BigQuery dataset (project.dataset) or gs:// path. Both argument forms are tried
    because the CLI has shipped both. `expect` is 'bigquery', 'bucket' or 'any'.

    The return code is checked and the SHAPE of the answer is validated, because neither is
    optional: a FAILED `wb` whose diagnostic or usage text goes to stdout would otherwise be
    accepted as a resolved id. WORKSPACE_CDR becomes truthy garbage, _fill substitutes it happily,
    the raise-on-unresolved contract never fires, and the whole thing surfaces many cells later as
    a bizarre BigQuery error.
    """
    wanted = {"bigquery": (_BQ_DATASET_RE,),
              "bucket": (_GCS_PATH_RE,),
              "any": (_BQ_DATASET_RE, _GCS_PATH_RE)}[expect]
    reasons = []
    for args in (["wb", "resource", "resolve", "--name", name],
                 ["wb", "resolve", "--name", name]):
        code, stdout, reason = _run_wb(args)
        if code != 0:
            reasons.append(reason)
            continue
        lines = [line.strip() for line in stdout.splitlines() if line.strip()]
        if len(lines) != 1:
            reasons.append(f"`wb` exited 0 but printed {len(lines)} lines where one id was expected")
            continue
        if not any(pattern.match(lines[0]) for pattern in wanted):
            reasons.append(f"`wb` exited 0 but returned {lines[0][:120]!r}, which is not a shape a "
                           f"{expect} resolve can return")
            continue
        return lines[0]
    print(f"wb_resolve({name!r}, expect={expect!r}) resolved nothing: {'; '.join(reasons)}")
    return None


# ---- CDR: the OMOP BigQuery dataset, cdrv9, resource name C2025Q4R6 ----
# Classic All of Us exposes os.environ["WORKSPACE_CDR"]; Verily Workbench 2.0 guarantees nothing, so
# fall back to the CLI resolve. The CDR lives in a DIFFERENT project (observed once as
# wb-silky-artichoke-2408) from the workspace's own, which is exactly why it is resolved every time
# and never assumed from GOOGLE_PROJECT.
if CONFIG_SKIP_NETWORK:
    WORKSPACE_CDR = globals()["WORKSPACE_CDR"]
    PREP_CDR = globals()["PREP_CDR"]
    print("already configured in this kernel: skipped the `wb` resolves for the CDR and the prep "
          "dataset. Set CONFIG_FORCE = True before %run to redo them.")
else:
    WORKSPACE_CDR = os.environ.get("WORKSPACE_CDR") or wb_resolve("C2025Q4R6", expect="bigquery")
    PREP_CDR = (os.environ.get("WORKSPACE_CDR_PREP")
                or wb_resolve("prep_C2025Q4R6", expect="bigquery"))

# ---- Derived dataset: where this study materializes its own table DAG ----
# The dataset NAME is ours and is fixed by the locked plan; the project qualifying it is resolved at
# runtime, so nothing here is a hardcoded cloud id. It must be created in the CDR's own location,
# see the location cell below.
DERIVED_DATASET = "spinewear_v1"
DERIVED = f"{GOOGLE_PROJECT}.{DERIVED_DATASET}" if GOOGLE_PROJECT else None

print("GOOGLE_PROJECT :", GOOGLE_PROJECT)
print("WORKSPACE_CDR  :", WORKSPACE_CDR)   # a dataset id is metadata, safe to display
print("PREP_CDR       :", PREP_CDR)
print("DERIVED        :", DERIVED)

# A None above is a stop condition, and THIS is where it stops. It used to be a comment only, so the
# next cell built a client with project=None, which does not fail: it silently falls back to the
# application default project, and the first hard failure downstream then blamed an unattached data
# collection, which is the wrong diagnosis for an unresolved name.
_unresolved = [label for label, value in (("GOOGLE_PROJECT", GOOGLE_PROJECT),
                                          ("WORKSPACE_CDR", WORKSPACE_CDR)) if not value]
if _unresolved:
    raise RuntimeError(
        f"{', '.join(_unresolved)} did not resolve, so nothing below may run. Do not paper over "
        f"this with a literal id. Resolve it by hand in a terminal, then export it:\n"
        f"    wb resource resolve --name C2025Q4R6\n"
        f"    echo $GOOGLE_PROJECT\n"
        f"GOOGLE_PROJECT is the project that PAYS, and a client built with project=None bills the "
        f"application default project instead, which is a different and much later failure."
    )
if not PREP_CDR:
    print("note: the prep dataset did not resolve. That is fatal only for a query using the {PREP} "
          "placeholder, and _fill raises there rather than substituting nothing.")

In [ ]:
# ---- BigQuery client and the free cost oracle ----
from google.cloud import bigquery

# Repeated rather than assumed: project=None does not fail, it silently bills the application
# default project. The cell above already stops on an unresolved name; this makes the client
# construction itself unable to paper over one if these cells are ever run out of order.
if not GOOGLE_PROJECT:
    raise RuntimeError("GOOGLE_PROJECT is unresolved, so there is no billing project to build a "
                       "client against. See the stop condition in the cell above.")

# project= is the BILLING project: the workspace project pays, even though the CDR sits elsewhere.
# Reused rather than rebuilt on a skipped run: constructing a client resolves application default
# credentials, which on a Workbench VM is a metadata-server round trip, and there is nothing to
# learn from repeating it once per notebook. The project is compared so a changed GOOGLE_PROJECT
# still gets a fresh client.
_cached_bq = globals().get("_bq")
if CONFIG_SKIP_NETWORK and getattr(_cached_bq, "project", None) == GOOGLE_PROJECT:
    _bq = _cached_bq
else:
    _bq = bigquery.Client(project=GOOGLE_PROJECT)

# BigQuery on-demand analysis list price, US, at the time of writing. One constant, one place to
# change it; nothing else in the project reads a price.
USD_PER_TIB = 6.25

# "GB" throughout this notebook means GiB (2**30 bytes), matching BigQuery's own billing unit (TiB),
# so a max_gb cap converts straight to a byte count. int() truncates, so a max_gb that is not a
# dyadic fraction lands the cap at most one byte BELOW the figure asked for: stricter, never looser,
# which is the only direction a cost cap is allowed to err in.
BYTES_PER_GIB = 1024 ** 3

# The only placeholders this notebook substitutes, named once so the residual scan can quote them
# back at whoever typed a fourth.
_KNOWN_PLACEHOLDERS = ("{CDR}", "{PREP}", "{DERIVED}")

# Identifier-shaped only, and deliberately so. A brace pair holding anything else (a JSON path, a
# struct literal, a formatted fragment inside a SQL string literal) is left alone, because this scan
# cannot tell a real placeholder from a legitimate brace inside a string literal, and a false stop
# condition on a valid query is worse than the miss. Every placeholder this project uses is
# {UPPER_SNAKE}, so the conservative pattern still catches all of them.
_RESIDUAL_PLACEHOLDER_RE = re.compile(r"\{[A-Za-z_][A-Za-z0-9_]*\}")


class UnresolvedPlaceholder(RuntimeError):
    """A query names a placeholder that is unresolved, unknown or misspelled.

    A class of its own because the write probe and every later diagnostic branch on it: nothing
    reached BigQuery, so this is never a permissions problem and must never collect the IAM
    diagnosis. It subclasses RuntimeError so existing `except RuntimeError` call sites still catch.
    """


def _fill(sql: str) -> str:
    """Substitute the dataset placeholders. Raise if any placeholder survives substitution.

    Silently substituting an empty string here is how a query ends up reading `.person` from nothing
    and failing three cells later with an unrelated-looking error. The residual scan is the other
    half of the same guarantee, and it is not hypothetical: {GOOGLE_PROJECT} is the exact spelling
    the locked plan uses when it says the DAG is materialized into `{GOOGLE_PROJECT}.spinewear_v1`,
    and {cdr} is one shift key away from {CDR}. Neither is substituted here, and without the scan
    both went straight to BigQuery as a syntax error three cells later.
    """
    for token, value in zip(_KNOWN_PLACEHOLDERS, (WORKSPACE_CDR, PREP_CDR, DERIVED)):
        if token in sql:
            if not value:
                raise UnresolvedPlaceholder(
                    f"{token} appears in this query but resolved to nothing. Resolve it first with "
                    f"wb resource resolve; do not substitute a literal id."
                )
            sql = sql.replace(token, value)
    residual = sorted(set(_RESIDUAL_PLACEHOLDER_RE.findall(sql)))
    if residual:
        raise UnresolvedPlaceholder(
            f"this query still contains {', '.join(residual)} after substitution, so it was not "
            f"sent. The only placeholders substituted here are "
            f"{', '.join(_KNOWN_PLACEHOLDERS)}, and they are case-sensitive. The derived dataset is "
            f"{{DERIVED}}, already qualified by the billing project, not "
            f"{{GOOGLE_PROJECT}}.spinewear_v1."
        )
    return sql


def dry_run_gb(sql: str) -> float:
    """Price a query before it runs. Returns the estimate in GiB and prints the dollar cost.

    A dry run is FREE: BigQuery plans the query and reports bytes without scanning anything. And
    BigQuery bills the sum of the COLUMNS REFERENCED, not the table, so every column of a
    multi-terabyte table can be priced at zero cost before a single byte is billed. That is the
    reason the whole cost design works: the frightening question is always answered for nothing
    first, and the minute-level Fitbit table is interrogated rather than guessed at.
    """
    cfg = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)
    job = _bq.query(_fill(sql), job_config=cfg)
    gb = (job.total_bytes_processed or 0) / BYTES_PER_GIB
    print(f"dry run estimate: {gb:,.3f} GB, about ${gb / 1024 * USD_PER_TIB:,.4f} "
          f"at ${USD_PER_TIB:,.2f} per TiB")
    return gb

In [ ]:
# ---- Session cost accumulator ----
# A dry run gives an estimate; this records what the jobs ACTUALLY billed, so the end-of-session
# handoff block in SESSION-LOG.md carries a real number instead of a guess. Every q_guarded call
# updates it. Bytes and dollars are job metadata, never participant data, so printing them is safe.
SESSION_COST = {"queries": 0, "bytes_billed": 0, "usd": 0.0, "log": []}


def _record_cost(bytes_billed: int | None, note: str = "") -> None:
    """Add one finished job's actual billed bytes to the running session total."""
    # A cache hit bills zero; otherwise BigQuery bills a 10 MB minimum per table scanned.
    billed = int(bytes_billed or 0)
    usd = billed / BYTES_PER_GIB / 1024 * USD_PER_TIB
    SESSION_COST["queries"] += 1
    SESSION_COST["bytes_billed"] += billed
    SESSION_COST["usd"] += usd
    SESSION_COST["log"].append({"note": note, "gb": billed / BYTES_PER_GIB, "usd": usd})


def session_cost_report() -> dict:
    """Print the running BigQuery spend for this session and return the accumulator."""
    gb = SESSION_COST["bytes_billed"] / BYTES_PER_GIB
    print(f"session BigQuery cost: {SESSION_COST['queries']} queries, "
          f"{gb:,.3f} GB billed, ${SESSION_COST['usd']:,.4f}")
    return SESSION_COST

In [ ]:
# ---- The only query path: dry run first, hard cap always ----
# Rule 2 of the project brief. No query executes without a printed byte estimate, and every query
# carries a maximum_bytes_billed cap so an over-budget query FAILS rather than bills.

DEFAULT_MAX_GB = 5.0  # roughly $0.03. Deliberately small: a query needing more must say so, and why.


class QueryCapExceeded(RuntimeError):
    """The dry-run estimate exceeded the cap, so nothing executed and nothing billed.

    A class of its own for the same reason as UnresolvedPlaceholder: a refusal by the cost cap is
    not a permissions problem, and the diagnostics downstream branch on the type rather than on
    "something raised". It subclasses RuntimeError so existing call sites still catch.
    """


def q_guarded(sql: str, *, max_gb: float, note: str = "") -> pd.DataFrame:
    """Dry-run, print the estimate, refuse if over cap, then execute under a hard byte cap.

    max_gb is BOTH the refusal threshold and the job's maximum_bytes_billed, so an estimate that
    turns out to be wrong (a stale statistic, a partition pruned differently at run time) costs
    nothing: the job fails with a quota error instead of billing past the cap.
    """
    sql = _fill(sql)
    gb = dry_run_gb(sql)                       # prints the estimate BEFORE anything executes
    if gb > max_gb:
        raise QueryCapExceeded(
            f"query refused, nothing executed: dry-run estimate {gb:,.3f} GB exceeds the cap of "
            f"{float(max_gb):,.3f} GB. note: {note or '(none)'}. Either narrow the columns "
            f"referenced or raise max_gb deliberately, with this number in hand."
        )
    cfg = bigquery.QueryJobConfig(
        maximum_bytes_billed=int(max_gb * BYTES_PER_GIB),
        use_query_cache=True,                  # a cache hit bills zero, so a repeated query is free
    )
    job = _bq.query(sql, job_config=cfg)
    df = job.result().to_dataframe()
    _record_cost(job.total_bytes_billed, note=note or (sql.strip().splitlines() or [""])[0][:60])
    return df


def q(sql: str, *, max_gb: float = DEFAULT_MAX_GB, note: str = "q, default cap") -> pd.DataFrame:
    """The ported q(), now a thin delegate to q_guarded so ported call sites keep working.

    The GWAS notebook's unguarded q() is GONE rather than kept alongside this one. A helper that can
    execute without a printed estimate and a hard cap is the helper that eventually gets typed at the
    end of a long session, and one such call against the minute-level Fitbit table is the entire
    project budget. Precisely: every HELPER in this notebook goes through the cap. The raw client
    `_bq` is still in the namespace, because dry_run_gb and q_guarded need it, and it is
    underscore-private to say that calling `_bq.query` yourself bypasses both the printed estimate
    and the cap. No module may.
    """
    return q_guarded(sql, max_gb=max_gb, note=note)

In [ ]:
# ---- Disclosure helpers: imported, never reimplemented ----
# pipeline/disclosure.py is the single source of truth for suppression and it is unit-tested there.
# A second copy of round20 living in this notebook is exactly how two different thresholds end up in
# one manuscript.
import sys
import pathlib


def _add_disclosure_to_path() -> pathlib.Path | None:
    """Locate disclosure.py next to this notebook even when the kernel's cwd is somewhere else."""
    candidates = []
    try:
        candidates.append(pathlib.Path(__file__).resolve().parent)   # defined when run as a script
    except NameError:
        pass
    cwd = pathlib.Path.cwd().resolve()
    candidates += [cwd, *cwd.parents]
    for directory in candidates:
        for cand in (directory / "disclosure.py", directory / "pipeline" / "disclosure.py"):
            if cand.is_file():
                if str(cand.parent) not in sys.path:
                    sys.path.insert(0, str(cand.parent))
                return cand
    return None


# Precedence is explicit, and the repo's own copy WINS. The locate helper used to run only in the
# `except ModuleNotFoundError` branch, so a different disclosure.py already on sys.path won
# silently and nothing said so. It runs first now, and the identity of what actually got imported
# is checked against it below rather than trusted.
_disclosure_local = _add_disclosure_to_path()

try:
    import disclosure as _disclosure
except ModuleNotFoundError:
    raise ModuleNotFoundError(
        "disclosure.py was not found next to this notebook or anywhere on sys.path. It is "
        "pipeline/disclosure.py in this repo. Nothing downstream may run without it, because "
        "every printed and exported surface goes through it."
    ) from None

_disclosure_src = getattr(_disclosure, "__file__", None) or "sys.path"
if _disclosure_local is not None and (
        pathlib.Path(_disclosure_src).resolve() != _disclosure_local.resolve()):
    raise RuntimeError(
        f"the imported disclosure module is {_disclosure_src}, but the copy belonging to this repo "
        f"is {_disclosure_local}. sys.path order cannot dislodge a module that is already imported, "
        f"so restart the kernel and run this notebook before anything else imports disclosure. Two "
        f"copies of round20 in one session means two thresholds in one manuscript."
    )
if _disclosure_local is None:
    print(f"warning: disclosure was imported from {_disclosure_src}, which is not next to this "
          f"notebook. Confirm it is this repo's copy before trusting a printed count.")

from disclosure import (MIN_CELL, ROUND_BASE, SUPPRESSED, DisclosureError, disclosable, round20,
                        is_suppressed, n_pct, prev, mean_sd, median_iqr, safe_show, safe_n,
                        safe_counts, suppress_frame, export_violations, md5_of_bytes, safe_export)

# The floor is policy, not a tunable. If it ever moves, the manuscript's suppression footnote and
# the flow figure's rounding footnote are both wrong.
#
# Two deliberate choices here. First, these are RAISES rather than asserts: `python -O` strips an
# assert, and a policy stop condition that vanishes under an optimization flag is worse than no
# check at all. Second, the floor is checked through the module's own predicate and its own
# constants rather than against a literal, because the export contract has verify.py grep the
# pipeline for a bare 20 in a comparison and fail the build on a hit.
if MIN_CELL != ROUND_BASE:
    raise DisclosureError(
        f"the suppression floor ({MIN_CELL}) and the rounding base ({ROUND_BASE}) have come apart. "
        f"The Methods sentence names one number for both, so two numbers here would need two "
        f"sentences and a reason. Fix disclosure.py; do not work around it in a notebook."
    )
if disclosable(MIN_CELL) or not disclosable(MIN_CELL + 1) or not disclosable(0):
    raise DisclosureError(
        f"disclosure.py's floor has moved: disclosable({MIN_CELL}) is {disclosable(MIN_CELL)}, "
        f"disclosable({MIN_CELL + 1}) is {disclosable(MIN_CELL + 1)}, disclosable(0) is "
        f"{disclosable(0)}. Policy is that a true count of MIN_CELL or fewer is suppressed, a count "
        f"strictly above it is disclosable, and an exact zero is disclosable and exported as 0."
    )
if round20(MIN_CELL) != SUPPRESSED:
    raise DisclosureError(
        f"round20({MIN_CELL}) returned {round20(MIN_CELL)!r} rather than the suppression marker "
        f"{SUPPRESSED!r}. A count at the floor must never render as a number."
    )

print(f"disclosure helpers loaded from {_disclosure_src}: MIN_CELL={MIN_CELL}, "
      f"ROUND_BASE={ROUND_BASE}, SUPPRESSED={SUPPRESSED!r}"
      f", DisclosureError, disclosable, round20, is_suppressed, n_pct, prev, mean_sd, median_iqr,"
      f" safe_show, safe_n, safe_counts, suppress_frame, export_violations, md5_of_bytes,"
      f" safe_export")

In [ ]:
# ---- Workspace resources this study actually needs ----
# The GWAS port resolved v9-genomics-folder and the controlled genomics bucket at this point. That
# cell is REPLACED rather than carried over: this study uses no genomic data at all, since the
# exposure is a procedure concept set and the outcome is Fitbit-derived, so resolving a callset would
# be an unused reference to a resource whose access is audited. What this study needs instead is a
# staging place for exports, plus the derived dataset resolved above.

# Why a status string and not just an empty list: `wb` not installed, a timeout, a permission error
# and "there is no bucket yet" all returned [], which makes them indistinguishable. 01_probe.py is
# specified to tell them apart, so it reads this rather than guessing from an empty list.
# Carried forward on a skipped run so the status still describes the resolve that actually ran.
WB_RESOURCE_LIST_STATUS = (globals().get("WB_RESOURCE_LIST_STATUS", "not attempted")
                           if globals().get("CONFIG_SKIP_NETWORK") else "not attempted")


def _list_resources() -> list[dict]:
    """Workspace resource names and types only. Metadata, not participant data, so safe to print."""
    global WB_RESOURCE_LIST_STATUS
    import json
    reasons = []
    for args in (["wb", "resource", "list", "--format=json"], ["wb", "resource", "list"]):
        code, stdout, reason = _run_wb(args)
        if code != 0:
            reasons.append(reason)
            continue
        txt = stdout.strip()
        if txt.startswith("["):
            try:
                rows = json.loads(txt)
            except ValueError as exc:
                reasons.append(f"`wb` exited 0 but its JSON did not parse ({exc})")
                continue
            WB_RESOURCE_LIST_STATUS = f"ok: {len(rows)} resources, machine-readable"
            return rows
        if txt:
            print(txt)          # human-readable table, read it by eye
            WB_RESOURCE_LIST_STATUS = "ok: human-readable table printed above, not machine-readable"
            return []
        reasons.append("`wb` exited 0 but printed nothing")
    WB_RESOURCE_LIST_STATUS = "failed: " + "; ".join(reasons)
    return []


if CONFIG_SKIP_NETWORK:
    WORKSPACE_BUCKET = globals()["WORKSPACE_BUCKET"]
    print("already configured in this kernel: skipped the workspace resource list and the bucket "
          "resolve; the status below is the one the first run recorded.")
else:
    _resources = _list_resources()
    _bucket_names = [r.get("id") or r.get("name") for r in _resources
                     if str(r.get("resourceType") or r.get("type") or "").upper().startswith("GCS")]
    WORKSPACE_BUCKET = wb_resolve(_bucket_names[0], expect="bucket") if _bucket_names else None

print("resource list    :", WB_RESOURCE_LIST_STATUS)
print("workspace bucket :", WORKSPACE_BUCKET or "none resolved yet (the bucket resource is Phase 1)")
print("derived dataset  :", DERIVED)
# The bucket is staging only. Results leave through safe_export as aggregate CSVs whose every count
# passes disclosable(), rounded, with an md5 stamp; nothing individual-level is ever written to it.

## The CDR's BigQuery location, resolved before anything is created

BigQuery **cannot join across locations**. If the CDR sits in the `US` multi-region and the derived
dataset were created in `us-central1`, every build query would fail, and the error reads like an IAM
problem rather than a geography problem. That misreading costs a session.

So the location is read from the CDR itself, which is a free metadata call, stored in `CDR_LOCATION`,
and `{GOOGLE_PROJECT}.spinewear_v1` is created to mirror it **exactly**. A dataset's location is
fixed at creation and cannot be changed afterwards, so a derived dataset that does not match the CDR
is a stop condition: it has to be dropped and recreated, not worked around.

In [ ]:
# ---- Resolve the CDR's BigQuery location. Free: metadata call, no bytes scanned. ----
if CONFIG_SKIP_NETWORK:
    CDR_LOCATION = globals()["CDR_LOCATION"]
    print("already configured in this kernel: skipped the CDR metadata call.")
else:
    try:
        CDR_LOCATION = _bq.get_dataset(WORKSPACE_CDR).location
    except Exception as exc:
        raise RuntimeError(
            f"could not read dataset metadata for {WORKSPACE_CDR!r}. The name itself resolved and "
            f"was shape-checked in the first cell, so this is not an unresolved-name failure: check "
            f"that the All of Us data collection is actually attached to this workspace and that "
            f"this account can read the dataset. An unattached collection fails here and at every "
            f"later step, and always looks like a permissions error. "
            f"({type(exc).__name__}: {exc})"
        ) from None

print("CDR location:", CDR_LOCATION, "(the derived dataset must mirror this exactly)")

In [ ]:
# ---- SELECT 1 write probe against {DERIVED} ----
# Fifteen free seconds that settle whether Phase 3 can materialize anything at all. Prior All of Us
# sessions rebuilt parquets every time because they had nowhere to write and used /tmp, which is the
# one location guaranteed to vanish with the compute disk.
if not (GOOGLE_PROJECT and DERIVED and CDR_LOCATION):
    raise RuntimeError("GOOGLE_PROJECT, DERIVED or CDR_LOCATION is unresolved; resolve them above "
                       "before probing, otherwise this cell diagnoses the wrong failure.")

# The client's own exception classes, so the diagnoses below branch on what actually failed rather
# than on "something raised". Layouts differ across releases, hence the fallback.
try:
    from google.api_core import exceptions as _gexc
except ImportError:
    from google.cloud import exceptions as _gexc


class _NeverRaised(Exception):
    """Stands in when a client release does not expose a permission class, so the tuple is valid."""


_NotFound = _gexc.NotFound
_PERMISSION_ERRORS = tuple(cls for cls in (getattr(_gexc, "Forbidden", None),
                                           getattr(_gexc, "Unauthorized", None),
                                           getattr(_gexc, "PermissionDenied", None))
                           if isinstance(cls, type)) or (_NeverRaised,)


class DatasetLocationMismatch(RuntimeError):
    """The derived dataset is not in the CDR's location. Geography, not permissions.

    Kept distinct so it never collects the IAM diagnosis below: the two failures look identical in a
    raw traceback and have nothing to do with each other.
    """


def _assert_derived_location(dataset) -> None:
    """Raise unless this dataset's location matches the CDR's, exactly and case-insensitively."""
    if (dataset.location or "").upper() != (CDR_LOCATION or "").upper():
        raise DatasetLocationMismatch(
            f"{DERIVED} exists in location {dataset.location} but the CDR is in {CDR_LOCATION}. "
            f"BigQuery cannot join across locations and a dataset's location cannot be changed after "
            f"creation, so this dataset must be dropped and recreated. Do not proceed."
        )


def _ensure_derived_dataset() -> str:
    """Create the derived dataset in the CDR's location if it is ABSENT. Returns what it did.

    Only NotFound means absent. A bare `except Exception` here used to send a transient 5xx, or a
    missing bigquery.datasets.get grant, down the create path, where create_dataset(exists_ok=True)
    succeeds silently against a dataset that may be in the WRONG location; the function then
    returned a plausible sentence naming the RIGHT one, skipping DatasetLocationMismatch entirely.
    The location is also read back after creating rather than reported from the request, because
    exists_ok=True can be satisfied by a dataset somebody else created a moment earlier.
    """
    try:
        existing = _bq.get_dataset(DERIVED)
    except _NotFound:
        ds = bigquery.Dataset(DERIVED)
        ds.location = CDR_LOCATION            # fixed at creation, so it has to be right the first time
        ds.description = "Derived tables for the spine wearable recovery-debt study (v1)."
        _bq.create_dataset(ds, exists_ok=True)
        landed = _bq.get_dataset(DERIVED)     # confirm the location it ACTUALLY landed in
        _assert_derived_location(landed)
        return f"created dataset {DERIVED} in location {landed.location}"
    except Exception as exc:
        raise RuntimeError(
            f"could not read dataset metadata for {DERIVED!r}, and the answer was not 'it does not "
            f"exist', so nothing was created. Creating on any error would mean creating against a "
            f"dataset that may already exist in the wrong location, which is the one failure this "
            f"cell exists to catch. ({type(exc).__name__}: {exc})"
        ) from None
    _assert_derived_location(existing)
    return f"dataset {DERIVED} already exists in location {existing.location}"


if CONFIG_SKIP_NETWORK:
    WRITE_PROBE_RESULT = globals()["WRITE_PROBE_RESULT"]
    print(f"already configured in this kernel: skipped the derived-dataset check and both write "
          f"probe jobs. First run reported: {WRITE_PROBE_RESULT}")
else:
    _billed_before = SESSION_COST["bytes_billed"]
    try:
        _what_it_did = _ensure_derived_dataset()
        q_guarded("CREATE OR REPLACE TABLE `{DERIVED}.write_probe` AS SELECT 1 AS ok",
                  max_gb=1.0, note="write probe, create")
        q_guarded("DROP TABLE `{DERIVED}.write_probe`", max_gb=1.0, note="write probe, drop")
        # Measured, not asserted. _record_cost captured job.total_bytes_billed for both probe jobs
        # a moment ago, so the honest thing to print is the number it captured.
        _probe_bytes = SESSION_COST["bytes_billed"] - _billed_before
        WRITE_PROBE_RESULT = (f"{_what_it_did}; wrote and dropped {DERIVED}.write_probe, "
                              f"{_probe_bytes:,} bytes billed across both probe jobs")
        print(f"write probe OK: {WRITE_PROBE_RESULT}")
    except DatasetLocationMismatch as exc:
        print(f"write probe HALTED: {exc}")
        print("Diagnosis: geography, not permissions. Nothing can join across locations, so drop this")
        print("dataset and recreate it in the CDR's location before anything else runs.")
        raise RuntimeError("derived dataset location does not match the CDR, see the diagnosis above") from None
    except UnresolvedPlaceholder as exc:
        print(f"write probe HALTED: {exc}")
        print("Diagnosis: an unresolved placeholder, not permissions. Nothing was sent to BigQuery,")
        print("so no grant is missing and nothing billed. Fix the name resolution in the first cell.")
        raise RuntimeError("write probe never ran: a placeholder is unresolved, see above") from None
    except QueryCapExceeded as exc:
        print(f"write probe HALTED: {exc}")
        print("Diagnosis: the cost cap refused the query, not permissions. Nothing executed and")
        print("nothing billed. A SELECT 1 probe priced above its cap means the estimate is not")
        print("measuring what this cell thinks it is; read the number before raising the cap.")
        raise RuntimeError("write probe refused by the cost cap, see the estimate above") from None
    except _PERMISSION_ERRORS as exc:
        print(f"write probe FAILED: {type(exc).__name__}: {exc}")
        print("Diagnosis: IAM, not VPC Service Controls. Both endpoints are inside the perimeter, so")
        print("this is an intra-perimeter write and the perimeter is not what refused it. The missing")
        print(f"grant is BigQuery Job User on {GOOGLE_PROJECT}, or Data Editor on {DERIVED}.")
        print("Ask for the role. Do not route around it by writing to /tmp or to a bucket: that is the")
        print("exact failure mode this dataset exists to fix.")
        raise RuntimeError("BigQuery write probe failed on permissions, see the diagnosis above") from None
    except Exception as exc:
        print(f"write probe FAILED: {type(exc).__name__}: {exc}")
        print("No diagnosis offered: this is not a location mismatch, not an unresolved placeholder,")
        print("not a cost-cap refusal and not a permissions error, so none of the four rehearsed")
        print("explanations applies. Believe the error above rather than a guess. Do not route around")
        print("it by writing to /tmp or to a bucket: that is the failure this dataset exists to fix.")
        raise RuntimeError("BigQuery write probe failed, see the error printed above") from None

In [ ]:
# ---- Software versions. These go into the meta block of results.json. ----
import re
import sys

import numpy as np
import statsmodels
import google.cloud.bigquery as _bq_pkg


def _version_tuple(text: str) -> tuple:
    return tuple(int(part) for part in re.findall(r"\d+", str(text))[:3])


# statsmodels 0.14 is the floor the analysis modules are written against. A silently older VM image
# would otherwise fail deep inside Phase 4, after the expensive queries, rather than here for free.
# Written as a raise rather than an assert: `python -O` strips an assert, and a stop condition that
# disappears under an optimization flag is worse than no check at all.
_STATSMODELS_FLOOR = (0, 14)
if _version_tuple(statsmodels.__version__) < _STATSMODELS_FLOOR:
    raise RuntimeError(
        f"statsmodels {statsmodels.__version__} is older than "
        f"{'.'.join(str(part) for part in _STATSMODELS_FLOOR)}. Stop condition: install a newer one "
        f"or change the plan. Do not work around it."
    )

try:
    from jupyter_client.kernelspec import KernelSpecManager
    _kernels = sorted(KernelSpecManager().find_kernel_specs())
except Exception:
    _kernels = []
R_KERNEL_VISIBLE = any(name.lower() == "r" or name.lower().startswith("ir") for name in _kernels)

SOFTWARE_VERSIONS = {
    "python": sys.version.split()[0],
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "statsmodels": statsmodels.__version__,
    "google-cloud-bigquery": _bq_pkg.__version__,
    "r-kernel-visible": R_KERNEL_VISIBLE,
}
for _name, _value in SOFTWARE_VERSIONS.items():
    print(f"{_name:<22} {_value}")

In [ ]:
# ---- Aggregate-only smoke test: confirms auth, the CDR resolve, and the guarded path end to end ----
# COUNT(*) with no filter is answered from table metadata, so the dry run prices it at zero.
if CONFIG_SKIP_NETWORK:
    PERSON_N_ROUNDED = globals()["PERSON_N_ROUNDED"]
    print("already configured in this kernel: skipped the person-count smoke test.")
else:
    _person_n = q_guarded("SELECT COUNT(*) AS n FROM `{CDR}.person`",
                          max_gb=1.0, note="startup smoke test, person row count")["n"].iloc[0]
    PERSON_N_ROUNDED = round20(_person_n)

print("person rows, rounded:", PERSON_N_ROUNDED)   # expect roughly 747,020 on cdrv9

# Bound rather than left as the trailing expression. As a bare expression an interactive run ALSO
# renders the returned dict, whose "log" carries the first 60 characters of every executed SQL.
# That is job metadata rather than participant data, but it is a second and noisier surface than
# the one printed line, and the printed line is the whole point.
_startup_cost = session_cost_report()

# Everything above succeeded, so a second `%run 00_config.ipynb` in this kernel may rebind the
# names and skip the network work. A run that raised anywhere above never reaches this line, so a
# half-configured session is never cached. CONFIG_FORCE is cleared here to keep it one-shot.
CONFIG_READY = True
CONFIG_FORCE = False

## Reuse in later notebooks and scripts

```python
%run 00_config.ipynb

df = q_guarded(
    "SELECT COUNT(*) AS n FROM `{CDR}.condition_occurrence`",
    max_gb=2.0,
    note="condition rows, sanity check",
)
print(round20(df["n"].iloc[0]))
```

- `q_guarded(sql, max_gb=..., note=...)` is the only query path any module may use. `q(sql)` still
  works and delegates to it under a 5 GiB default cap, so there is no unguarded **helper** here.
  The raw client is `_bq`, underscore-private and used only by `dry_run_gb` and `q_guarded`;
  calling `_bq.query` yourself bypasses both the printed estimate and the cap, and no module may.
- `{CDR}`, `{PREP}` and `{DERIVED}` are substituted for you. An unresolved one raises rather than
  substituting an empty string, and so does any brace token left over after substitution, so a
  misspelled `{cdr}` or the plan's own `{GOOGLE_PROJECT}` spelling stops here rather than arriving
  at BigQuery as a syntax error. `{DERIVED}` is this study's own dataset, already qualified by the
  billing project, and it lives in the CDR's location.
- The second and later `%run 00_config.ipynb` in one kernel rebinds every name and **skips the
  network work**: the `wb` resolves, the dataset metadata calls, the write probe and the person
  count. Each skipped cell prints what it skipped. To force the full run again, after changing a
  workspace resource or to re-probe write access, set `CONFIG_FORCE = True` immediately before the
  `%run` (or export `SPINEWEAR_CONFIG_FORCE=1`). It is one-shot and is cleared on success.
- Price anything unfamiliar with `dry_run_gb(sql)` first. It is free, and it prices the columns
  referenced rather than the table, so even the minute-level Fitbit table costs nothing to interrogate.
- Every export goes through `safe_export` from `disclosure.py`. Never surface a raw row. If
  individual-level data has to be inspected, the human does it with the automation paused.
- Before deleting the environment, run `session_cost_report()` and paste the number into the handoff
  block in `SESSION-LOG.md`. Then delete the environment, and verify the Apps tab is empty.